In [ ]:
```xml
<VSCode.Cell language="markdown">
# 📋 Data Governance & Lineage Guide

**Managing data quality, ownership, lineage tracking, compliance, and data stewardship at scale**

This guide covers:
- Data governance frameworks
- Data cataloging and metadata management
- Data lineage tracking
- Data quality metrics and monitoring
- Data ownership and stewardship
- Compliance and regulatory requirements
- Data classification and sensitivity levels
- Audit trails and versioning
</VSCode.Cell>

<VSCode.Cell language="markdown">
## 1. Data Governance Framework

### Data Catalog
```python
from dataclasses import dataclass
from typing import Dict, List, Set, Optional
from datetime import datetime
from enum import Enum

class DataClassification(Enum):
    """Data sensitivity classification"""
    PUBLIC = "public"
    INTERNAL = "internal"
    CONFIDENTIAL = "confidential"
    RESTRICTED = "restricted"

class DataOwnershipTier(Enum):
    """Ownership responsibility levels"""
    PRIMARY = "primary"
    SECONDARY = "secondary"
    TERTIARY = "tertiary"

@dataclass
class DataAsset:
    """Catalog entry for data asset"""
    id: str
    name: str
    description: str
    classification: DataClassification
    owner: str
    steward: str
    location: str  # Storage path/URI
    schema: Dict  # Column info
    created_at: datetime
    last_modified: datetime
    quality_score: float  # 0.0-1.0
    tags: Set[str]
    lineage_in: List[str]  # Upstream sources
    lineage_out: List[str]  # Downstream uses
    
    def is_sensitive(self) -> bool:
        """Check if asset is sensitive"""
        return self.classification in [DataClassification.CONFIDENTIAL, DataClassification.RESTRICTED]
    
    def get_audit_trail(self) -> Dict:
        """Get audit information"""
        return {
            'asset_id': self.id,
            'owner': self.owner,
            'steward': self.steward,
            'classification': self.classification.value,
            'created': self.created_at.isoformat(),
            'modified': self.last_modified.isoformat(),
            'quality_score': self.quality_score
        }

class DataCatalog:
    """Central data asset registry"""
    
    def __init__(self):
        self.assets: Dict[str, DataAsset] = {}
        self.ownership_map: Dict[str, List[str]] = {}  # owner -> asset IDs
        self.lineage_graph: Dict[str, List[str]] = {}  # asset -> downstream
    
    def register_asset(self, asset: DataAsset):
        """Register data asset in catalog"""
        self.assets[asset.id] = asset
        
        if asset.owner not in self.ownership_map:
            self.ownership_map[asset.owner] = []
        
        self.ownership_map[asset.owner].append(asset.id)
        
        # Build lineage graph
        for upstream in asset.lineage_in:
            if upstream not in self.lineage_graph:
                self.lineage_graph[upstream] = []
            self.lineage_graph[upstream].append(asset.id)
    
    def get_asset(self, asset_id: str) -> Optional[DataAsset]:
        """Retrieve asset from catalog"""
        return self.assets.get(asset_id)
    
    def find_by_owner(self, owner: str) -> List[DataAsset]:
        """Find all assets owned by person"""
        asset_ids = self.ownership_map.get(owner, [])
        return [self.assets[aid] for aid in asset_ids]
    
    def search_by_classification(self, classification: DataClassification) -> List[DataAsset]:
        """Find assets by sensitivity level"""
        return [a for a in self.assets.values() if a.classification == classification]
    
    def trace_lineage_upstream(self, asset_id: str, depth: int = 5) -> List[str]:
        """Find all upstream data sources"""
        if depth == 0:
            return []
        
        asset = self.get_asset(asset_id)
        if not asset:
            return []
        
        upstream = list(asset.lineage_in)
        
        for source in asset.lineage_in:
            upstream.extend(self.trace_lineage_upstream(source, depth - 1))
        
        return list(set(upstream))  # Deduplicate
    
    def trace_lineage_downstream(self, asset_id: str, depth: int = 5) -> List[str]:
        """Find all downstream consumers"""
        if depth == 0:
            return []
        
        downstream = self.lineage_graph.get(asset_id, [])
        
        for target in downstream:
            downstream.extend(self.trace_lineage_downstream(target, depth - 1))
        
        return list(set(downstream))  # Deduplicate

# Usage
catalog = DataCatalog()

asset1 = DataAsset(
    id="user_transactions",
    name="User Transactions",
    description="Daily transaction records",
    classification=DataClassification.CONFIDENTIAL,
    owner="data_team",
    steward="john_doe",
    location="s3://data/transactions",
    schema={"user_id": "int", "amount": "decimal", "timestamp": "datetime"},
    created_at=datetime.now(),
    last_modified=datetime.now(),
    quality_score=0.95,
    tags={"pii", "financial", "daily"},
    lineage_in=[],
    lineage_out=["user_summary_report", "fraud_detection_model"]
)

catalog.register_asset(asset1)
```

### Quality Metrics
```python
class DataQualityMetrics:
    """Track data quality dimensions"""
    
    def __init__(self):
        self.completeness_score = 0.0  # % non-null
        self.accuracy_score = 0.0  # % matches reference
        self.consistency_score = 0.0  # Format/type conformance
        self.timeliness_score = 0.0  # Recency of data
        self.uniqueness_score = 0.0  # % unique (for unique fields)
        self.validity_score = 0.0  # % passes validation rules
    
    def calculate_overall_quality(self) -> float:
        """Calculate composite quality score"""
        scores = [
            self.completeness_score,
            self.accuracy_score,
            self.consistency_score,
            self.timeliness_score,
            self.uniqueness_score,
            self.validity_score
        ]
        
        return sum(scores) / len(scores)
    
    def flag_quality_issues(self, threshold: float = 0.8) -> List[str]:
        """Identify quality dimensions below threshold"""
        issues = []
        
        if self.completeness_score < threshold:
            issues.append(f"Low completeness: {self.completeness_score:.1%}")
        if self.accuracy_score < threshold:
            issues.append(f"Low accuracy: {self.accuracy_score:.1%}")
        if self.consistency_score < threshold:
            issues.append(f"Low consistency: {self.consistency_score:.1%}")
        if self.timeliness_score < threshold:
            issues.append(f"Stale data: {self.timeliness_score:.1%}")
        
        return issues

class DataQualityMonitor:
    """Monitor and enforce data quality"""
    
    def __init__(self, catalog: DataCatalog):
        self.catalog = catalog
        self.quality_history: Dict[str, List[DataQualityMetrics]] = {}
        self.slas: Dict[str, float] = {}  # asset_id -> min_quality_score
    
    def set_sla(self, asset_id: str, min_quality: float):
        """Set quality SLA for asset"""
        self.slas[asset_id] = min_quality
    
    def assess_asset(self, asset_id: str, metrics: DataQualityMetrics) -> bool:
        """Assess asset quality against SLA"""
        overall = metrics.calculate_overall_quality()
        
        if asset_id not in self.quality_history:
            self.quality_history[asset_id] = []
        
        self.quality_history[asset_id].append(metrics)
        
        sla = self.slas.get(asset_id, 0.8)
        is_compliant = overall >= sla
        
        if not is_compliant:
            print(f"⚠️  Quality SLA violation for {asset_id}: {overall:.1%} < {sla:.1%}")
        
        return is_compliant
    
    def get_quality_trend(self, asset_id: str) -> List[float]:
        """Get quality trend over time"""
        if asset_id not in self.quality_history:
            return []
        
        return [m.calculate_overall_quality() for m in self.quality_history[asset_id]]
```
</VSCode.Cell>

<VSCode.Cell language="markdown">
## 2. Data Lineage & Audit Trails

### Lineage Tracking
```python
@dataclass
class LineageNode:
    """Node in data lineage graph"""
    id: str
    asset_id: str
    operation: str  # EXTRACT, TRANSFORM, LOAD, QUERY, etc.
    timestamp: datetime
    parameters: Dict
    status: str  # SUCCESS, FAILURE, PARTIAL
    duration_ms: float

class DataLineageTracker:
    """Track data lineage and transformations"""
    
    def __init__(self):
        self.lineage_events: List[LineageNode] = []
        self.lineage_graph: Dict[str, List[str]] = {}  # source -> targets
    
    def record_extraction(self, source_asset_id: str, target_asset_id: str, params: Dict = None):
        """Record data extraction"""
        node = LineageNode(
            id=f"extract_{source_asset_id}_{target_asset_id}",
            asset_id=source_asset_id,
            operation="EXTRACT",
            timestamp=datetime.now(),
            parameters=params or {},
            status="SUCCESS",
            duration_ms=0
        )
        
        self.lineage_events.append(node)
        self._add_edge(source_asset_id, target_asset_id)
    
    def record_transformation(self, source_assets: List[str], target_asset_id: str, transform: str, params: Dict = None):
        """Record data transformation"""
        for source in source_assets:
            node = LineageNode(
                id=f"transform_{source}_{target_asset_id}",
                asset_id=source,
                operation="TRANSFORM",
                timestamp=datetime.now(),
                parameters={'transform': transform, **params or {}},
                status="SUCCESS",
                duration_ms=0
            )
            
            self.lineage_events.append(node)
            self._add_edge(source, target_asset_id)
    
    def _add_edge(self, source: str, target: str):
        """Add edge to lineage graph"""
        if source not in self.lineage_graph:
            self.lineage_graph[source] = []
        
        if target not in self.lineage_graph[source]:
            self.lineage_graph[source].append(target)
    
    def get_lineage_report(self, asset_id: str) -> Dict:
        """Generate lineage report for asset"""
        return {
            'asset_id': asset_id,
            'events_count': len([e for e in self.lineage_events if e.asset_id == asset_id]),
            'transformations': len([e for e in self.lineage_events if e.operation == "TRANSFORM" and e.asset_id == asset_id]),
            'total_duration_ms': sum(e.duration_ms for e in self.lineage_events if e.asset_id == asset_id),
            'recent_operations': [e.operation for e in self.lineage_events[-5:] if e.asset_id == asset_id]
        }

class AuditTrail:
    """Maintain complete audit trail of data operations"""
    
    def __init__(self):
        self.audit_log: List[Dict] = []
    
    def log_access(self, user: str, asset_id: str, operation: str, timestamp: datetime = None):
        """Log data access"""
        self.audit_log.append({
            'timestamp': timestamp or datetime.now(),
            'user': user,
            'asset_id': asset_id,
            'operation': operation,
            'status': 'completed'
        })
    
    def log_modification(self, user: str, asset_id: str, change: str, before: Dict, after: Dict, timestamp: datetime = None):
        """Log data modification"""
        self.audit_log.append({
            'timestamp': timestamp or datetime.now(),
            'user': user,
            'asset_id': asset_id,
            'operation': 'MODIFY',
            'change_description': change,
            'before': before,
            'after': after
        })
    
    def get_user_activity(self, user: str) -> List[Dict]:
        """Get all activities by user"""
        return [log for log in self.audit_log if log['user'] == user]
    
    def get_asset_history(self, asset_id: str) -> List[Dict]:
        """Get complete history of asset"""
        return [log for log in self.audit_log if log['asset_id'] == asset_id]
    
    def compliance_report(self, start_date: datetime, end_date: datetime) -> Dict:
        """Generate compliance report"""
        period_logs = [log for log in self.audit_log 
                      if start_date <= log['timestamp'] <= end_date]
        
        return {
            'period': f"{start_date.date()} to {end_date.date()}",
            'total_operations': len(period_logs),
            'unique_users': len(set(log['user'] for log in period_logs)),
            'operations_by_type': self._count_by_field(period_logs, 'operation'),
            'modifications': len([log for log in period_logs if log['operation'] == 'MODIFY'])
        }
    
    def _count_by_field(self, logs: List[Dict], field: str) -> Dict[str, int]:
        """Count logs by field value"""
        counts = {}
        for log in logs:
            key = log.get(field)
            counts[key] = counts.get(key, 0) + 1
        return counts
```
</VSCode.Cell>

<VSCode.Cell language="markdown">
## 3. Compliance & Policy Enforcement

### Data Classification & Access Control
```python
class DataPolicy:
    """Define data governance policies"""
    
    def __init__(self, name: str, classification: DataClassification):
        self.name = name
        self.classification = classification
        self.allowed_operations: Set[str] = set()
        self.allowed_roles: Set[str] = set()
        self.retention_days: int = 365
        self.encryption_required: bool = False
        self.audit_required: bool = False
    
    def allow_operation(self, operation: str, role: str):
        """Allow operation for role"""
        self.allowed_operations.add(operation)
        self.allowed_roles.add(role)
    
    def can_access(self, user_role: str, operation: str) -> bool:
        """Check if access is allowed"""
        return user_role in self.allowed_roles and operation in self.allowed_operations

class PolicyEnforcer:
    """Enforce data governance policies"""
    
    def __init__(self):
        self.policies: Dict[str, DataPolicy] = {}
    
    def register_policy(self, classification: DataClassification, policy: DataPolicy):
        """Register governance policy"""
        self.policies[classification.value] = policy
    
    def can_access(self, user_role: str, asset: DataAsset, operation: str) -> bool:
        """Check access permissions"""
        policy = self.policies.get(asset.classification.value)
        
        if not policy:
            return True  # No policy = allow
        
        if operation not in policy.allowed_operations:
            return False
        
        if user_role not in policy.allowed_roles:
            return False
        
        return True
    
    def enforce_access(self, user_role: str, asset: DataAsset, operation: str) -> bool:
        """Enforce access and log"""
        allowed = self.can_access(user_role, asset, operation)
        
        if not allowed:
            print(f"❌ Access denied: {user_role} cannot {operation} {asset.id}")
        
        return allowed
```

### Regulatory Compliance
```python
class ComplianceRequirement(Enum):
    """Regulatory frameworks"""
    GDPR = "gdpr"
    CCPA = "ccpa"
    HIPAA = "hipaa"
    SOC2 = "soc2"
    PCI_DSS = "pci_dss"

class ComplianceChecker:
    """Validate compliance requirements"""
    
    def __init__(self, catalog: DataCatalog):
        self.catalog = catalog
        self.requirements: Dict[ComplianceRequirement, Dict] = {}
    
    def add_requirement(self, framework: ComplianceRequirement, rules: Dict):
        """Add compliance rules"""
        self.requirements[framework] = rules
    
    def check_gdpr_compliance(self, asset: DataAsset) -> List[str]:
        """Check GDPR compliance"""
        violations = []
        
        if asset.is_sensitive() and not asset.owner:
            violations.append("No data owner assigned (GDPR requires accountability)")
        
        # Check retention
        rules = self.requirements.get(ComplianceRequirement.GDPR, {})
        max_retention = rules.get('max_retention_years', 5)
        
        if asset.lineage_in:  # Has upstream sources
            violations.append("Personal data sources must be documented (GDPR Article 14)")
        
        return violations
    
    def check_pci_dss_compliance(self, asset: DataAsset) -> List[str]:
        """Check PCI DSS compliance for payment data"""
        violations = []
        
        if 'payment' in asset.name.lower() or 'card' in asset.name.lower():
            # Sensitive payment data
            if asset.classification == DataClassification.PUBLIC:
                violations.append("Payment data cannot be public (PCI DSS 3.2.1)")
        
        return violations
    
    def generate_compliance_report(self, framework: ComplianceRequirement) -> Dict:
        """Generate compliance report"""
        violations_by_asset = {}
        
        for asset_id, asset in self.catalog.assets.items():
            if framework == ComplianceRequirement.GDPR:
                violations = self.check_gdpr_compliance(asset)
            elif framework == ComplianceRequirement.PCI_DSS:
                violations = self.check_pci_dss_compliance(asset)
            else:
                violations = []
            
            if violations:
                violations_by_asset[asset_id] = violations
        
        return {
            'framework': framework.value.upper(),
            'total_assets': len(self.catalog.assets),
            'compliant_assets': len(self.catalog.assets) - len(violations_by_asset),
            'violations': violations_by_asset
        }
```
</VSCode.Cell>

<VSCode.Cell language="markdown">
## 4. Data Governance Checklist

✅ **Catalog & Metadata**
- [ ] Data asset registry created
- [ ] Ownership assigned
- [ ] Metadata standardized
- [ ] Tags and descriptions added
- [ ] Sensitive data marked

✅ **Quality Management**
- [ ] Quality metrics defined
- [ ] SLAs established
- [ ] Monitoring alerts configured
- [ ] Quality issues tracked
- [ ] Remediation processes documented

✅ **Lineage Tracking**
- [ ] Data sources documented
- [ ] Transformations tracked
- [ ] Downstream uses identified
- [ ] Impact analysis possible
- [ ] Lineage graph visualized

✅ **Compliance**
- [ ] Regulatory requirements identified
- [ ] Policies documented
- [ ] Access controls implemented
- [ ] Audit trails maintained
- [ ] Compliance reports generated

✅ **Governance Operations**
- [ ] Data stewards assigned
- [ ] Review cycles scheduled
- [ ] Escalation paths defined
- [ ] Training program established
- [ ] Metrics dashboard built
</VSCode.Cell>
```